# 73 DPO check on validation (CPU runtime)

Method "cost first, then accuracy": (1) a cost-guided search finds the cheapest correct query known for every train question; (2) the current model (`rl_v1_it8`) answers every train question 8 times; (3) preference pairs: chosen = the cheap query, rejected = the model's own costlier or wrong answer; (4) DPO from the current model; (5) RL from the DPO model, with the cheapest known query as the cost yardstick; (6) the model is chosen on validation by a fixed rule, then dev and mini-dev are scored once.

Decision D3. The DPO models are scored on validation only (dev and mini-dev wait for the end) and compared with the current model by the rule fixed before training:
1. a model is allowed unless its EX is significantly lower than the current model's (McNemar, 0.05);
2. among allowed models, the lowest cost relative to the current model on the questions both answer correctly wins, if below 1; otherwise the current model is kept.

If the current model is kept here, the cost did not fall and RL (step 5) cannot help: we stop and write up. **Runtime > Change runtime type > CPU.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
EXPORT = "sft_v1"                   # SFT export: its 4 held-out train databases are the validation set
CURRENT = "rl_v1_it8"               # our current model (RL, kept on validation): the start and the reference
SEARCH = "search_v1"                # 1. cost-guided search, all train questions -> cheap targets
OWN = "own_v1"                      # 2. the current model's own answers, 8 per train question
PAIRS = "pairs_v2"                  # 3. preference pairs (main)
DPO = "dpo_v2"                      # 4. DPO from CURRENT, 2 epochs: dpo_v2_e1 (after epoch 1) and dpo_v2
DPO_RUNS = [f"{DPO}_e1", DPO]
RL = "rl_v2"                        # 5. RL from the DPO model kept on validation: rl_v2_it8 and rl_v2
RL_RUNS = [f"{RL}_it8", RL]
# Ablations (DPO only, same settings as the kept DPO model): where the chosen and rejected answers come from.
ABLATIONS = {"gold": "--chosen gold",      # chosen = gold: the generated queries are not used at all
             "pool": "--chosen pool",      # chosen = the first pool's cheapest query: no search
             "off": "--rejected pool"}     # rejected = other models' pool queries, not the model's own
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
VAL_DBS = ",".join(json.load(open(f"{R}/{EXPORT}/export_meta.json"))["holdout_db_ids"])
def done(run):
    return os.path.exists(f"{R}/{run}/adapter_meta.json")
print("validation databases:", VAL_DBS)
def evaluate_runs(runs, dev=True):
    for r in runs:
        method = json.load(open(f"{R}/{r}/adapter_meta.json")).get("method", "model")
        !python -m effsql.train eval-val --run {r}
        if dev:
            !effsql evaluate --pred {R}/{r}/predictions_dev.jsonl --split dev --timing none --run {r}_dev --model {MODEL} --method {method}
            !effsql evaluate --pred {R}/{r}/predictions_minidev.jsonl --split minidev --timing quick --run {r}_minidev --model {MODEL} --method {method}

### Stage the validation and dev databases

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev

### 1. Score the DPO models on validation (resumable)

In [ ]:
evaluate_runs(DPO_RUNS, dev=False)

### 2. The rule

`cost ratio`: geometric mean of the model's cost / the current model's on the questions both answer correctly (below 1 = cheaper); `cheaper` / `costlier`: those questions where it is at least 10% cheaper / costlier, with a sign test `p`.

In [ ]:
!python -m effsql.train select-cost --ref {CURRENT} --runs {','.join(DPO_RUNS)} --out {DPO}_choice

### 3. Training log (pair loss and accuracy on the validation pairs)

In [ ]:
for r in open(f"{R}/{DPO}/val_log.jsonl"):
    v = json.loads(r)
    print(v["step"], {k: round(v[k], 3) for k in v if k.startswith(("loss", "reward_acc", "margin"))})

### Done: send me the output, then disconnect the runtime